# Re-run: beyond the grade (RQ3)

From the benchmark repository's committed outputs: `feedback-board/out/pushback.json` (the pushback
effect per model), `maths-process/out/process.json` (credit for working, per model and version of each
problem) and `feedback-discordant/out/discordant.json` (feedback told a wrong grade). Counts come from the
intervals.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grading-benchmark-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


SOURCE_ENV, SOURCE_NAME = "GBENCH_DIR", "ai-grading-benchmark"   # set when a release's code is another repository


def source_repo():
    """The repository a release's code and committed outputs are published in (a release whose paper cites
    it rather than a copied code package): $SOURCE_ENV, or a checkout named SOURCE_NAME beside this
    notebook or beside a folder above it."""
    here = Path.cwd()
    places = [Path(os.environ[SOURCE_ENV])] if SOURCE_ENV and os.environ.get(SOURCE_ENV) else []
    places += [p / SOURCE_NAME for p in [here, *here.parents]]
    for d in places:
        if d.is_dir() and (d / "out").is_dir():
            return d
    raise FileNotFoundError(
        f"Could not find the {SOURCE_NAME} repository this release's code is published in. Clone it at the "
        f"commit the release file names, then set the environment variable {SOURCE_ENV} to the clone, or "
        f"run this notebook from a folder beside it.")


def load_source_json(relpath):
    """A committed output of that repository, by its path there (cited in the paper as code/<path>)."""
    return json.loads((source_repo() / relpath).read_text(encoding="utf-8"))


print("Source repository:", shown(source_repo()))

Source repository: ../../../../../../ai-grading-benchmark


In [2]:
from decimal import Decimal, ROUND_HALF_UP

def p(x, d=3):
    return f"{Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP):.{d}f}"

pb = load_source_json("feedback-board/out/pushback.json")["models"]
yields = sum(v["pushback"]["ci95"][0] > 0 for v in pb.values())
print(yields, "of", len(pb), "models raise the grade when the student only disagrees")
assert (yields, len(pb)) == (17, 22)

gr = [g for g in load_source_json("maths-process/out/process.json")["graders"] if g["grader"] in pb]
reads = [g for g in gr if g["reasoning_ci95"][0] > 0.5]
print(len(reads), "of", len(gr), "read the working (interval above 0.5 of 4)")
assert (len(reads), len(gr)) == (20, 22)
slip = [g["mean_credit"]["slip"] for g in reads]
print("credit for right working with a slip at the end:", p(min(slip)), "to", p(max(slip)), "; difference", p(max(slip) - min(slip)))
assert (p(min(slip)), p(max(slip)), p(max(slip) - min(slip))) == ("1.550", "3.415", "1.865")

17 of 22 models raise the grade when the student only disagrees
20 of 22 read the working (interval above 0.5 of 4)
credit for right working with a slip at the end: 1.550 to 3.415 ; difference 1.865


In [3]:
fd = load_source_json("feedback-discordant/out/discordant.json")["rows"]
follows = [r for r in fd if r["ci95"][0] > 0 and r["verdict"] == "TONE FOLLOWS THE TOLD GRADE"]
print(len(follows), "of", len(fd), "models: feedback tone follows a wrong told grade")
assert (len(follows), len(fd)) == (22, 22)
ge = [r["model"] for r in fd if r["told_grade_slope"] >= r["work_slope_blind"]]
print("a wrong grade moves the tone at least as much as the work does for", len(ge), "models")
assert len(ge) == 8

22 of 22 models: feedback tone follows a wrong told grade
a wrong grade moves the tone at least as much as the work does for 8 models
